# 01 - Module 1: Lakehouse basics, Delta, time travel, `MERGE`

Hands-on companion for [`module-1-lakehouse-basics.md`](../module-1-lakehouse-basics.md).

**Prerequisite:** you have run `00_setup.ipynb` and the `bi_course` schema
exists with all three tables populated.

**SSIS parallels you will see in this notebook:**

- `_delta_log/` ↔ SQL Server transaction log.
- `DESCRIBE HISTORY` ↔ SSISDB execution log + parsed transaction log.
- `VERSION AS OF` ↔ restoring a database backup, but in one query.
- `MERGE INTO` ↔ the SSIS *Slowly Changing Dimension* wizard, in 6 lines.


## 1. Inspect the table


In [ ]:
%sql
USE workspace.bi_course;
DESCRIBE EXTENDED workspace.bi_course.fact_sales;


## 2. An atomic `UPDATE`

Order `1003` was mis-priced. Fix it.


In [ ]:
%sql
UPDATE workspace.bi_course.fact_sales
SET    revenue = 1150.00
WHERE  order_id = 1003;


## 3. An accidental `INSERT` (broken referential integrity)

The classic 'flaky SSIS Lookup' bug: a row with non-existent
`product_id`/`region_id` slipped through.


In [ ]:
%sql
INSERT INTO workspace.bi_course.fact_sales VALUES
    (9999, DATE'2026-01-21', 99, 99, 1, 0.00);


## 4. Audit with `DESCRIBE HISTORY`


In [ ]:
%sql
DESCRIBE HISTORY workspace.bi_course.fact_sales;


## 5. Time travel - read a past version

Replace `2` below with whatever version `DESCRIBE HISTORY` shows for the
commit just **before** the bad insert.


In [ ]:
%sql
SELECT COUNT(*) AS rows_at_v2
FROM workspace.bi_course.fact_sales VERSION AS OF 2;


In [ ]:
%sql
SELECT COUNT(*) AS rows_one_minute_ago
FROM workspace.bi_course.fact_sales
TIMESTAMP AS OF current_timestamp() - INTERVAL 1 MINUTE;


## 6. Roll back the bad insert

Two options - use whichever fits your scenario:


In [ ]:
%sql
-- Option A: surgical delete.
DELETE FROM workspace.bi_course.fact_sales WHERE order_id = 9999;


In [ ]:
%sql
-- Option B: restore the whole table to a known-good version.
-- (Comment out Option A first, or this is a no-op.)
-- RESTORE TABLE workspace.bi_course.fact_sales TO VERSION AS OF 2;


## 7. Stretch - `MERGE INTO` (SCD Type 1)

**This single statement is what the SSIS *Slowly Changing Dimension*
wizard generates.** It replaces an OLE DB Source + Lookup +
Conditional Split + OLE DB Command + OLE DB Destination.


In [ ]:
%sql
CREATE OR REPLACE TEMP VIEW v_product_updates AS
SELECT * FROM VALUES
    (3, 'Wireless Mouse', 'Accessories',  29.00),  -- price bump
    (5, 'Monitor 27"',    'Displays',    310.00),  -- price bump
    (7, 'Webcam HD',      'Accessories',  60.00)   -- new product
AS t(product_id, product_name, category, list_price);


In [ ]:
%sql
MERGE INTO workspace.bi_course.dim_product AS tgt
USING v_product_updates                    AS src
   ON tgt.product_id = src.product_id
WHEN MATCHED     THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;


In [ ]:
%sql
SELECT * FROM workspace.bi_course.dim_product ORDER BY product_id;


Two updated rows (3 and 5) and one new row (7)? Module 1 complete.
On to `02_bi_with_sql.ipynb`.
